In [ ]:
# ================== CLEAN SETUP ==================
import os, warnings, logging, re, random
os.environ["WANDB_DISABLED"] = "true"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ================== IMPORTS ==================
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import nltk
from nltk.corpus import wordnet

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from torch.nn import CrossEntropyLoss

nltk.download("wordnet")

# ================== TEXT CLEANING ==================
def clean_text(text):
    # Convert non-string inputs to empty string
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+", "", text)   # remove URLs
    text = re.sub(r"@\w+", "", text)      # remove mentions
    text = re.sub(r"#", "", text)         # remove hashtags
    text = re.sub(r"\s+", " ", text)      # remove extra spaces
    return text.strip().lower()

# ================== AUGMENTATION ==================
def synonym_replacement(sentence, n=1):
    words = sentence.split()
    new_words = words.copy()
    random_word_list = list(set([w for w in words if len(wordnet.synsets(w)) > 0]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = wordnet.synsets(random_word)
        if not synonyms:
            continue
        synonym_words = [lemma.name().replace("_", " ")
                         for syn in synonyms for lemma in syn.lemmas()
                         if lemma.name() != random_word]
        if synonym_words:
            synonym = random.choice(synonym_words)
            new_words = [synonym if w == random_word else w for w in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return " ".join(new_words)

def random_deletion(sentence, p=0.1):
    words = sentence.split()
    if len(words) == 1:
        return sentence
    new_words = [w for w in words if random.uniform(0, 1) > p]
    if not new_words:
        return random.choice(words)
    return " ".join(new_words)

def augment_text(text):
    if random.random() < 0.5:
        return synonym_replacement(text, n=1)
    else:
        return random_deletion(text, p=0.1)

# ================== LOAD DATA ==================
df = pd.read_csv("Dataset_1_2_3_4.csv")

if df["label"].dtype == "object":
    label_mapping = {"Non-Hate": 0, "Hate": 1}
    df["label"] = df["label"].map(label_mapping)

df["label"] = df["label"].astype(int)
df["text"] = df["text"].apply(clean_text)

# Augment training data (30%)
augmented_rows = []
for text, label in zip(df["text"], df["label"]):
    if random.random() < 0.3:
        augmented_text = augment_text(text)
        augmented_rows.append({"text": augmented_text, "label": label})

df_aug = pd.DataFrame(augmented_rows)
df = pd.concat([df, df_aug]).reset_index(drop=True)

# Split dataset
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# ================== CLASS WEIGHTS ==================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# ================== TOKENIZER ==================
MODEL_NAME = "sentence-transformers/LaBSE"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=tokenizer, max_len=192):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextDataset(train_texts, train_labels)
test_dataset  = TextDataset(test_texts, test_labels)

# ================== MODEL ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ================== TRAINING ARGS ==================
training_args = TrainingArguments(
    output_dir="./results_labse",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,   # 🔹 Changed from 5 to 10
    weight_decay=0.01,
    warmup_steps=300,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# ================== TRAIN ==================
trainer.train()

# ================== EVALUATE ==================
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=-1)

print("\n✅ Test Accuracy:", accuracy_score(test_labels, y_pred))
print("\n✅ Classification Report:\n", classification_report(test_labels, y_pred, target_names=["Non-Hate", "Hate"]))

[nltk_data] Downloading package wordnet to /root/nltk_data...


tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Using device: cuda


model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/LaBSE and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.586900,0.506108,0.761649
2,0.415300,0.421405,0.815412
3,0.234500,0.445973,0.842294
4,0.113100,0.531554,0.846774
5,0.054100,0.739255,0.863799
6,0.027400,0.844584,0.863799
7,0.014300,0.978426,0.865143
8,0.006500,1.039133,0.868280
9,0.007800,1.038646,0.866039
10,0.003200,1.069079,0.866487



✅ Test Accuracy: 0.8682795698924731

✅ Classification Report:
               precision    recall  f1-score   support

    Non-Hate       0.87      0.92      0.89      1360
        Hate       0.86      0.79      0.82       872

    accuracy                           0.87      2232
   macro avg       0.87      0.85      0.86      2232
weighted avg       0.87      0.87      0.87      2232

